# 00 Jalankan Pipeline REVISED


In [ ]:
# Jalur REVISED dan pembaca hasil praproses.
from pathlib import Path
import os, sys
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
_explicit=os.environ.get('CARE_KM_ROOT') or os.environ.get('CARE_KM_PAPER_ROOT')
_options=([Path(_explicit)] if _explicit else []) + [Path('/content/drive/MyDrive/REVISED'),
    Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd(),Path.cwd(),Path.cwd().parent]
_ROOT_REVISED=next((p.resolve() for p in _options if all((p/d).is_dir() for d in ('DATA','LIB','PYTHON'))),None)
if _ROOT_REVISED is None:raise FileNotFoundError('Letakkan DATA, LIB, PYTHON di MyDrive/REVISED.')
sys.path.insert(0,str(_ROOT_REVISED/'LIB'))
# Jangan memakai modul versi lama yang masih tersimpan di memori Colab.
import importlib
_loaded=sys.modules.get('pipeline_paths')
if True:
    for _mod in ('pipeline_paths','pipeline_data','pipeline_runner','pipeline_cache','experiment_lib','care_km','generalization_data'):
        sys.modules.pop(_mod,None)
    importlib.invalidate_caches()
# Muat ulang pustaka proyek dari kode sumber yang sekarang.
sys.dont_write_bytecode=True
_library_root=(_ROOT_REVISED/'LIB').resolve()
for _cached in (_library_root/'__pycache__').glob('*.pyc'):
    if _cached.resolve().is_relative_to(_library_root) and not _cached.is_symlink():
        _cached.unlink()
from pipeline_paths import configure,stage_dir,lms_data,pendikom_data,artifact_path,record_resample
configure(_ROOT_REVISED)


## Tahap tambahan

Master menjalankan tahap 01–24. Tahap 25–33 dijalankan dari notebook masing-masing.


In [ ]:
_P25_COMPAT_SOURCE = '"""Allow one independent notebook 25 beside the original 00–24 notebooks.\n\nOnly inventory and input dependency accounting are extended. No experiment\ncode, parameters, expected outputs, or execution order are changed.\n"""\nimport ast as _p25_ast\nimport json as _p25_json\nfrom pathlib import Path as _p25_Path\nimport pipeline_runner as _p25_runner\nimport pipeline_cache as _p25_cache\n\n_P25_OPTIONAL_STEMS = {\'25_Perbandingan_Initializer_Terintegrasi\', \'25_Verifikasi_Baseline_BK_Modes_2025\', \'26_Pengaruh_Initializer_Profil_Semantik\', \'27_Perbandingan_Pembentukan_Profil\', \'28_Referensi_Supervised_ILS\', \'29_Sensitivitas_Jumlah_Cluster_K\', \'30_Direct_Assignment_vs_Classifier\', \'31_Benchmark_Runtime_dan_Failure_Audit\', \'32_Transportabilitas_Antarkursus_dan_Waktu\', \'33_Analisis_Gabungan_Hasil\'}\n_P25_COMPAT_TAG = \'optional-stage25-compatibility\'\n\n\ndef _p25_carrier(payload, checksum):\n    return ("_P25_COMPAT_SOURCE = " + repr(payload) + "\\n"\n            + "_P25_COMPAT_SHA256 = " + repr(checksum) + "\\n"\n            + "import hashlib as _p25_boot_hashlib\\n"\n            + "if _p25_boot_hashlib.sha256(_P25_COMPAT_SOURCE.encode(\'utf-8\')).hexdigest() != _P25_COMPAT_SHA256:\\n"\n            + "    raise RuntimeError(\'Sel kompatibilitas tahap 25 berubah; gunakan versi paket yang lengkap.\')\\n"\n            + "exec(compile(_P25_COMPAT_SOURCE, \'optional_stage25_compatibility\', \'exec\'), globals())\\n")\n\n\ndef _p25_install(payload, checksum):\n    if getattr(_p25_runner, \'_optional_stage25_installed\', False):\n        return\n    original_input_state = _p25_cache.input_state\n    expected_carrier = _p25_ast.dump(_p25_ast.parse(_p25_carrier(payload, checksum)))\n\n    def validated_inventory(root):\n        scheduled = _p25_runner.notebook_order()\n        expected = set(scheduled) | {\'00_Jalankan_Pipeline_REVISED\'}\n        actual = {path.stem for path in (_p25_Path(root) / \'PYTHON\').glob(\'*.ipynb\')}\n        independent = _P25_OPTIONAL_STEMS & actual\n        missing, extra = expected - actual, actual - expected - independent\n        if missing or extra:\n            raise RuntimeError(\'Daftar notebook belum sesuai. Kurang: \' + str(sorted(missing))\n                               + \'; tambahan belum terdaftar: \' + str(sorted(extra))\n                               + \'. Notebook 25/26/27/28/29/30/31/32/33 yang terdaftar dijalankan langsung di Colab.\')\n        if len(scheduled) != len(set(scheduled)):\n            raise RuntimeError(\'Daftar tahap utama memiliki notebook berulang.\')\n        if sorted(int(name.split(\'_\', 1)[0]) for name in expected) != list(range(25)):\n            raise RuntimeError(\'Nomor tahap utama harus berurutan dari 00 hingga 24.\')\n        return scheduled\n\n    def compatible_input_state(stem, root, hashes, runtime, options):\n        result = original_input_state(stem, root, hashes, runtime, options)\n        state = result[\'manifest\']\n        # Notebook 25 is independent: it is not an input to any original stage.\n        for optional_stem in _P25_OPTIONAL_STEMS:\n            state[\'notebooks\'].pop(\'PYTHON/\' + optional_stem + \'.ipynb\', None)\n        master_name = \'PYTHON/00_Jalankan_Pipeline_REVISED.ipynb\'\n        master = _p25_json.loads((_p25_Path(root) / master_name).read_text(encoding=\'utf-8-sig\'))\n        original_cells = []\n        recognized = 0\n        for cell in master[\'cells\']:\n            if cell[\'cell_type\'] != \'code\':\n                continue\n            source = \'\'.join(cell[\'source\'])\n            if _P25_COMPAT_TAG in cell.get(\'metadata\', {}).get(\'tags\', []):\n                if _p25_ast.dump(_p25_ast.parse(source)) != expected_carrier:\n                    raise RuntimeError(\'Sel kompatibilitas master tidak sesuai versi yang diverifikasi.\')\n                recognized += 1\n            else:\n                original_cells.append(source)\n        if recognized != 1:\n            raise RuntimeError(\'Master memerlukan tepat satu sel kompatibilitas tahap 25.\')\n        # Retain the original master-cell fingerprint, including its actual options.\n        # All original LIB files, scientific notebook cells, data, parameters,\n        # environments and output inventories remain checked by the old cache.\n        state[\'notebooks\'][master_name] = _p25_cache.digest_json(original_cells)\n        result[\'sha256\'] = _p25_cache.digest_json(state)\n        return result\n\n    _p25_runner.validate_notebook_inventory = validated_inventory\n    _p25_runner.input_state = compatible_input_state\n    _p25_cache.input_state = compatible_input_state\n    _p25_runner._optional_stage25_installed = True\n\n\n_p25_install(_P25_COMPAT_SOURCE, _P25_COMPAT_SHA256)\nprint(\'Master utama: tahap 01–24. Notebook 25 sampai 33 diterima di PYTHON dan dijalankan langsung di Colab.\')\n'
_P25_COMPAT_SHA256 = '79f65055951a92452d3284e2e3507eea70119a48536908f3e689e10736bcca17'
import hashlib as _p25_boot_hashlib
if _p25_boot_hashlib.sha256(_P25_COMPAT_SOURCE.encode('utf-8')).hexdigest() != _P25_COMPAT_SHA256:
    raise RuntimeError('Sel kompatibilitas tahap 25 berubah; gunakan versi paket yang lengkap.')
exec(compile(_P25_COMPAT_SOURCE, 'optional_stage25_compatibility', 'exec'), globals())


In [ ]:
JALANKAN_PRAPROSES = False
JALANKAN_BENCHMARK = True
LANJUTKAN = True
from pipeline_runner import run_pipeline
status=run_pipeline(_ROOT_REVISED,preprocess=JALANKAN_PRAPROSES,
                    benchmarks=JALANKAN_BENCHMARK,resume=LANJUTKAN)
display(__import__('pandas').DataFrame(status)[['notebook','status']])
